# Notebook 09 — Flood Susceptibility Model Validation

## Objective

This notebook evaluates the predictive performance of the flood susceptibility models developed in **Notebook 08**.

The validation focuses on the ability of the models to identify flood-prone locations in **spatially withheld data**, rather than evaluating the final mapping models on the same observations used for training.

The supervised modelling years are:

- **2003** — DFO flood reference
- **2025** — Global Flood Monitor (GFM) flood reference

The **2014** dataset is not used as a supervised validation target because it was previously determined that a sufficiently defensible flood-reference label could not be established.

## Validation Strategy

Notebook 08 established a spatial validation framework using:

- **5 km spatial blocks**
- a geographically contiguous **NW region as the withheld test region**
- **3-fold spatial cross-validation** for model development
- block-level separation between training and withheld evaluation data

This notebook uses that established design to evaluate the final model performance.

## Models

The models carried forward from Notebook 08 are:

1. **Random Forest (RF)** — selected primary susceptibility model
2. **XGBoost (XGB)** — secondary comparison model

The primary validation focus is on the continuous susceptibility scores produced by these models.

## Important Principle

The final RF mapping models generated in Notebook 08 were trained using all available labelled cells and are therefore **not used to claim independent validation performance**.

Instead, validation uses the models/data associated with the spatially withheld evaluation design established during model development.

## Outputs

Validation results will be saved under:

```text
outputs/analysis_output/

## 09.1 — Locate and Import Validation Inputs

### Purpose

The first step is to load the data and saved model artefacts required for validation.

No validation metric or performance analysis is performed in this step.

We will first confirm that the required inputs from Notebook 08 are available and can be loaded correctly.

### Required Inputs

For each supervised year, we need:

- the modelling dataset containing the predictor variables and flood target;
- the spatial train/test assignment established in Notebook 08;
- the tuned evaluation model used for the withheld spatial test;
- the predictor-variable list and associated configuration.

The final all-data Random Forest mapping models are **not used as the validation models**, because they were trained using all labelled modelling cells.

### Output Organization

Validation tables will be written to:

```text
outputs/analysis_output/

In [1]:
# ============================================================
# 09.1 — Locate and Import Validation Inputs
# ============================================================

from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

# Adjust this only if PROJECT_ROOT is not already defined
PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = PROJECT_ROOT / "data"
MODEL_ROOT = PROJECT_ROOT / "models"

ANALYSIS_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analysis_output"
FIGURE_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "figures" / "notebook_09"

ANALYSIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("NOTEBOOK 09 — VALIDATION INPUT SETUP")
print("=" * 70)

print(f"\nProject root:")
print(PROJECT_ROOT)

print(f"\nAnalysis output:")
print(ANALYSIS_OUTPUT_DIR)

print(f"\nFigure output:")
print(FIGURE_OUTPUT_DIR)


# ------------------------------------------------------------
# 2. Locate modelling datasets
# ------------------------------------------------------------

MODELLING_DIR = PROJECT_ROOT / "outputs" / "tables" / "modelling_dataset"

modelling_2003_path = MODELLING_DIR / "modelling_2003_250m.csv"
modelling_2025_path = MODELLING_DIR / "modelling_2025_250m.csv"

print("\n" + "-" * 70)
print("MODELLING DATASETS")
print("-" * 70)

print(f"2003: {modelling_2003_path}")
print(f"      {'FOUND' if modelling_2003_path.exists() else 'MISSING'}")

print(f"2025: {modelling_2025_path}")
print(f"      {'FOUND' if modelling_2025_path.exists() else 'MISSING'}")


# ------------------------------------------------------------
# 3. Load modelling datasets
# ------------------------------------------------------------

if not modelling_2003_path.exists():
    raise FileNotFoundError(
        f"2003 modelling dataset not found:\n{modelling_2003_path}"
    )

if not modelling_2025_path.exists():
    raise FileNotFoundError(
        f"2025 modelling dataset not found:\n{modelling_2025_path}"
    )

modelling_2003 = pd.read_csv(modelling_2003_path)
modelling_2025 = pd.read_csv(modelling_2025_path)

print("\nLoaded modelling datasets:")
print(f"2003 shape: {modelling_2003.shape}")
print(f"2025 shape: {modelling_2025.shape}")


# ------------------------------------------------------------
# 4. Locate saved models
# ------------------------------------------------------------

rf_2003_path = MODEL_ROOT / "rf_final_mapping_2003.joblib"
rf_2025_path = MODEL_ROOT / "rf_final_mapping_2025.joblib"

xgb_2003_path = MODEL_ROOT / "xgb_tuned_evaluation_2003.joblib"
xgb_2025_path = MODEL_ROOT / "xgb_tuned_evaluation_2025.joblib"

print("\n" + "-" * 70)
print("SAVED MODELS")
print("-" * 70)

model_paths = {
    "RF 2003": rf_2003_path,
    "RF 2025": rf_2025_path,
    "XGB 2003": xgb_2003_path,
    "XGB 2025": xgb_2025_path,
}

for name, path in model_paths.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name:<12}: {status:<8} {path}")


# ------------------------------------------------------------
# 5. Load RF mapping models if available
# ------------------------------------------------------------

rf_mapping_models = {}

if rf_2003_path.exists():
    rf_mapping_models[2003] = joblib.load(rf_2003_path)

if rf_2025_path.exists():
    rf_mapping_models[2025] = joblib.load(rf_2025_path)

print("\nLoaded RF mapping models:")
for year, model in rf_mapping_models.items():
    print(
        f"{year}: {type(model).__name__} | "
        f"n_estimators={getattr(model, 'n_estimators', 'N/A')}"
    )


# ------------------------------------------------------------
# 6. Load XGBoost evaluation models if available
# ------------------------------------------------------------

xgb_evaluation_models = {}

if xgb_2003_path.exists():
    xgb_evaluation_models[2003] = joblib.load(xgb_2003_path)

if xgb_2025_path.exists():
    xgb_evaluation_models[2025] = joblib.load(xgb_2025_path)

print("\nLoaded XGBoost evaluation models:")
for year, model in xgb_evaluation_models.items():
    print(
        f"{year}: {type(model).__name__}"
    )


# ------------------------------------------------------------
# 7. Inspect dataset columns
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("DATASET COLUMNS")
print("-" * 70)

print("\n2003 columns:")
print(list(modelling_2003.columns))

print("\n2025 columns:")
print(list(modelling_2025.columns))


# ------------------------------------------------------------
# 8. Basic target check
# ------------------------------------------------------------

for year, df in {
    2003: modelling_2003,
    2025: modelling_2025
}.items():

    if "Flood" in df.columns:
        print(f"\n{year} flood target:")
        print(df["Flood"].value_counts(dropna=False).sort_index())
    else:
        print(f"\nWARNING: Flood column not found for {year}")


print("\n" + "=" * 70)
print("STEP 09.1 — INPUT LOADING COMPLETE")
print("=" * 70)

NOTEBOOK 09 — VALIDATION INPUT SETUP

Project root:
d:\Projects\GeoAI-Flood-Susceptibility

Analysis output:
d:\Projects\GeoAI-Flood-Susceptibility\outputs\analysis_output

Figure output:
d:\Projects\GeoAI-Flood-Susceptibility\outputs\figures\notebook_09

----------------------------------------------------------------------
MODELLING DATASETS
----------------------------------------------------------------------
2003: d:\Projects\GeoAI-Flood-Susceptibility\outputs\tables\modelling_dataset\modelling_2003_250m.csv
      FOUND
2025: d:\Projects\GeoAI-Flood-Susceptibility\outputs\tables\modelling_dataset\modelling_2025_250m.csv
      FOUND

Loaded modelling datasets:
2003 shape: (47523, 15)
2025 shape: (43076, 15)

----------------------------------------------------------------------
SAVED MODELS
----------------------------------------------------------------------
RF 2003     : FOUND    d:\Projects\GeoAI-Flood-Susceptibility\models\rf_final_mapping_2003.joblib
RF 2025     : FOUND    d:

## 09.2 — Reconstruct the Withheld Spatial Test Set

### Purpose

Notebook 08 established the final spatial validation design using a **5 km spatial-block framework**.

For both 2003 and 2025, the **NW spatial region** was reserved as the withheld test region. All modelling cells belonging to blocks in this region were excluded from model training and retained for final spatial evaluation.

This step reconstructs that withheld test set from the modelling datasets.

### Why this is necessary

Flood-susceptibility observations located close to one another can be spatially correlated. A conventional random cell-level split could therefore place neighbouring cells in both training and testing datasets, allowing spatial information to leak between the two datasets.

The block-based split prevents the same 5 km spatial block from contributing cells to both training and testing.

### Validation Design

- Spatial block size: **5 km**
- Test region: **NW**
- Training regions: **NE, SW, SE, and the remaining non-NW blocks**
- Test blocks are completely excluded from training
- The same spatial holdout definition is applied separately to 2003 and 2025

### Step 2 Goal

For each year, create:

- a training-cell dataset;
- a withheld NW test-cell dataset;
- the corresponding predictor matrix;
- the corresponding flood target vector.

Before any performance metrics are calculated, the split will be checked to confirm:

1. all cells are assigned exactly once;
2. training and test sets contain no overlapping spatial blocks;
3. both training and test sets retain the expected flood observations;
4. the resulting cell counts are consistent with the spatial split established in Notebook 08.

**No model performance metrics are calculated in this step.**

In [6]:
# ============================================================================
# 09.2 — RECONSTRUCT EXACT NOTEBOOK 08 SPATIAL TRAIN-TEST SPLIT
# ============================================================================

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box

print("=" * 75)
print("09.2 — RECONSTRUCTING EXACT NOTEBOOK 08 SPATIAL SPLIT")
print("=" * 75)


# ----------------------------------------------------------------------------
# 1. Exact configuration used in Notebook 08
# ----------------------------------------------------------------------------

GRID_LEFT = 463250.0
GRID_BOTTOM = 2938000.0

VALIDATION_BLOCK_SIZE = 5000  # 5 km

TEST_REGION = "NW"

print(f"\nGrid origin X      : {GRID_LEFT}")
print(f"Grid origin Y      : {GRID_BOTTOM}")
print(f"Block size         : {VALIDATION_BLOCK_SIZE / 1000:.1f} km")
print(f"Test region        : {TEST_REGION}")


# ----------------------------------------------------------------------------
# 2. Recreate the EXACT spatial-block construction from Notebook 08
# ----------------------------------------------------------------------------

def create_validation_blocks(df, block_size):
    """
    Recreate the exact spatial-block construction used in Notebook 08.

    Each modelling cell is assigned to a 5 km block using the common
    modelling-grid origin. A rectangular polygon is then created for
    each block. Flood information is summarized at block level.
    """

    temp = df.copy()

    # ------------------------------------------------------------------------
    # Assign modelling cells to spatial blocks
    # ------------------------------------------------------------------------

    temp["block_x"] = np.floor(
        (temp["x_utm"] - GRID_LEFT) / block_size
    ).astype(int)

    temp["block_y"] = np.floor(
        (temp["y_utm"] - GRID_BOTTOM) / block_size
    ).astype(int)

    temp["block_id"] = (
        temp["block_x"].astype(str)
        + "_"
        + temp["block_y"].astype(str)
    )

    # ------------------------------------------------------------------------
    # Summarize cells within each block
    # ------------------------------------------------------------------------

    block_summary = (
        temp.groupby(
            ["block_id", "block_x", "block_y"],
            as_index=False
        )
        .agg(
            total_cells=("Flood", "size"),
            flood_cells=("Flood", "sum")
        )
    )

    # ------------------------------------------------------------------------
    # Create the SAME rectangular block geometries as Notebook 08
    # ------------------------------------------------------------------------

    geometries = []

    for _, row in block_summary.iterrows():

        xmin = (
            GRID_LEFT
            + row["block_x"] * block_size
        )

        ymin = (
            GRID_BOTTOM
            + row["block_y"] * block_size
        )

        xmax = xmin + block_size
        ymax = ymin + block_size

        geometries.append(
            box(
                xmin,
                ymin,
                xmax,
                ymax
            )
        )

    # ------------------------------------------------------------------------
    # Convert to GeoDataFrame
    # ------------------------------------------------------------------------

    blocks_gdf = gpd.GeoDataFrame(
        block_summary,
        geometry=geometries,
        crs="EPSG:32644"
    )

    # ------------------------------------------------------------------------
    # Flood-containing blocks
    # ------------------------------------------------------------------------

    blocks_gdf["contains_flood"] = (
        blocks_gdf["flood_cells"] > 0
    )

    return blocks_gdf


# ----------------------------------------------------------------------------
# 3. Recreate 5 km blocks for both years
# ----------------------------------------------------------------------------

validation_blocks = {}

for year, df in [
    (2003, modelling_2003),
    (2025, modelling_2025)
]:

    blocks = create_validation_blocks(
        df,
        VALIDATION_BLOCK_SIZE
    )

    validation_blocks[year] = blocks

    print("\n" + "-" * 75)
    print(f"{year} — SPATIAL BLOCKS")
    print("-" * 75)

    print(f"Total blocks     : {len(blocks):,}")
    print(
        f"Flood-containing : "
        f"{int(blocks['contains_flood'].sum()):,}"
    )
    print(
        f"Non-flood-only   : "
        f"{int((~blocks['contains_flood']).sum()):,}"
    )


# ----------------------------------------------------------------------------
# 4. Calculate block centroids EXACTLY as Notebook 08
# ----------------------------------------------------------------------------

for year in [2003, 2025]:

    blocks = validation_blocks[year].copy()

    # IMPORTANT:
    # Notebook 08 uses polygon geometry centroids,
    # NOT mean modelling-cell coordinates.

    centroids = blocks.geometry.centroid

    blocks["centroid_x"] = centroids.x
    blocks["centroid_y"] = centroids.y

    # Block class
    blocks["block_class"] = np.where(
        blocks["flood_cells"] > 0,
        "flood",
        "non_flood"
    )

    validation_blocks[year] = blocks

    print("\n" + "-" * 75)
    print(f"{year} — BLOCK CENTROID CHECK")
    print("-" * 75)

    print(
        f"Median centroid X : "
        f"{blocks['centroid_x'].median():.3f}"
    )

    print(
        f"Median centroid Y : "
        f"{blocks['centroid_y'].median():.3f}"
    )


# ----------------------------------------------------------------------------
# 5. EXACT spatial-region assignment from Notebook 08
# ----------------------------------------------------------------------------

def assign_spatial_regions(blocks):
    """
    Exact spatial-region assignment used in Notebook 08.

    Regions are based only on median block-centroid coordinates.
    Flood labels are NOT used for region assignment.
    """

    blocks = blocks.copy()

    median_x = blocks["centroid_x"].median()
    median_y = blocks["centroid_y"].median()

    blocks["spatial_region"] = np.select(
        [
            # NW
            (blocks["centroid_x"] <= median_x) &
            (blocks["centroid_y"] >= median_y),

            # NE
            (blocks["centroid_x"] > median_x) &
            (blocks["centroid_y"] >= median_y),

            # SW
            (blocks["centroid_x"] <= median_x) &
            (blocks["centroid_y"] < median_y),

            # SE
            (blocks["centroid_x"] > median_x) &
            (blocks["centroid_y"] < median_y)
        ],
        [
            "NW",
            "NE",
            "SW",
            "SE"
        ],
        default="Unknown"
    )

    return blocks


for year in [2003, 2025]:

    validation_blocks[year] = assign_spatial_regions(
        validation_blocks[year]
    )

    blocks = validation_blocks[year]

    print("\n" + "-" * 75)
    print(f"{year} — SPATIAL REGION DISTRIBUTION")
    print("-" * 75)

    print(
        blocks["spatial_region"]
        .value_counts()
        .sort_index()
        .to_string()
    )


# ----------------------------------------------------------------------------
# 6. Select the same NW holdout used in Notebook 08
# ----------------------------------------------------------------------------

spatial_splits = {}

for year in [2003, 2025]:

    blocks = validation_blocks[year].copy()

    blocks["dataset"] = np.where(
        blocks["spatial_region"] == TEST_REGION,
        "test",
        "train"
    )

    spatial_splits[year] = blocks


# ----------------------------------------------------------------------------
# 7. Transfer block-level split to modelling cells
# ----------------------------------------------------------------------------

def assign_cell_split(df, spatial_blocks_df):
    """
    Recreate the exact Notebook 08 cell-level train/test assignment.
    """

    temp = df.copy()

    # Same block calculation as Notebook 08
    temp["block_x"] = np.floor(
        (temp["x_utm"] - GRID_LEFT)
        / VALIDATION_BLOCK_SIZE
    ).astype(int)

    temp["block_y"] = np.floor(
        (temp["y_utm"] - GRID_BOTTOM)
        / VALIDATION_BLOCK_SIZE
    ).astype(int)

    temp["block_id"] = (
        temp["block_x"].astype(str)
        + "_"
        + temp["block_y"].astype(str)
    )

    # Block-level train/test assignment
    block_assignment = spatial_blocks_df[
        ["block_id", "dataset"]
    ].copy()

    temp = temp.merge(
        block_assignment,
        on="block_id",
        how="left",
        validate="many_to_one"
    )

    return temp


modelling_2003_split = assign_cell_split(
    modelling_2003,
    spatial_splits[2003]
)

modelling_2025_split = assign_cell_split(
    modelling_2025,
    spatial_splits[2025]
)


# ----------------------------------------------------------------------------
# 8. Verify no cells were left unassigned
# ----------------------------------------------------------------------------

for year, df in [
    (2003, modelling_2003_split),
    (2025, modelling_2025_split)
]:

    missing_split = df["dataset"].isna().sum()

    print(
        f"\n{year} — Unassigned cells: "
        f"{missing_split:,}"
    )

    assert missing_split == 0, (
        f"{year}: some modelling cells have no train/test assignment."
    )


# ----------------------------------------------------------------------------
# 9. Final train-test statistics
# ----------------------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL NOTEBOOK 08 SPATIAL TRAIN-TEST SPLIT")
print("=" * 75)

for year, df in [
    (2003, modelling_2003_split),
    (2025, modelling_2025_split)
]:

    train_df = df[
        df["dataset"] == "train"
    ]

    test_df = df[
        df["dataset"] == "test"
    ]

    print("\n" + "-" * 75)
    print(f"YEAR {year}")
    print("-" * 75)

    print(f"Full cells        : {len(df):,}")
    print(f"Training cells    : {len(train_df):,}")
    print(f"Testing cells     : {len(test_df):,}")

    print(
        f"Training flood    : "
        f"{int(train_df['Flood'].sum()):,}"
    )

    print(
        f"Training nonflood : "
        f"{int((train_df['Flood'] == 0).sum()):,}"
    )

    print(
        f"Testing flood     : "
        f"{int(test_df['Flood'].sum()):,}"
    )

    print(
        f"Testing nonflood  : "
        f"{int((test_df['Flood'] == 0).sum()):,}"
    )


# ----------------------------------------------------------------------------
# 10. Verify zero spatial-block overlap
# ----------------------------------------------------------------------------

print("\n" + "=" * 75)
print("SPATIAL BLOCK OVERLAP CHECK")
print("=" * 75)

for year in [2003, 2025]:

    blocks = spatial_splits[year]

    train_blocks = set(
        blocks.loc[
            blocks["dataset"] == "train",
            "block_id"
        ]
    )

    test_blocks = set(
        blocks.loc[
            blocks["dataset"] == "test",
            "block_id"
        ]
    )

    overlap = train_blocks.intersection(
        test_blocks
    )

    print(f"\n{year}")
    print(f"Training blocks   : {len(train_blocks):,}")
    print(f"Testing blocks    : {len(test_blocks):,}")
    print(f"Overlapping blocks: {len(overlap):,}")

    assert len(overlap) == 0


print("\n" + "=" * 75)
print("✓ STEP 09.2 — EXACT NOTEBOOK 08 SPATIAL SPLIT RECONSTRUCTED")
print("=" * 75)

09.2 — RECONSTRUCTING EXACT NOTEBOOK 08 SPATIAL SPLIT

Grid origin X      : 463250.0
Grid origin Y      : 2938000.0
Block size         : 5.0 km
Test region        : NW

---------------------------------------------------------------------------
2003 — SPATIAL BLOCKS
---------------------------------------------------------------------------
Total blocks     : 142
Flood-containing : 13
Non-flood-only   : 129

---------------------------------------------------------------------------
2025 — SPATIAL BLOCKS
---------------------------------------------------------------------------
Total blocks     : 141
Flood-containing : 66
Non-flood-only   : 75

---------------------------------------------------------------------------
2003 — BLOCK CENTROID CHECK
---------------------------------------------------------------------------
Median centroid X : 495750.000
Median centroid Y : 2970500.000

---------------------------------------------------------------------------
2025 — BLOCK CENTROID CHEC

## 09.3 Validation Model Preparation

### Purpose

This step identifies the model objects required for evaluation on the spatially withheld NW region.

The final mapping Random Forest models saved in Notebook 08 were trained using the complete labelled datasets and are therefore not used to claim withheld spatial validation performance.

Validation predictions must instead come from models trained only on the spatial training portion, with the NW blocks completely withheld during evaluation.

The Random Forest model is the primary model, while XGBoost is retained as the secondary comparison model.

No validation metrics are calculated in this step.

In [7]:
# ============================================================================
# 09.3 — LOCATE SAVED VALIDATION / EVALUATION MODELS
# ============================================================================

from pathlib import Path

print("=" * 75)
print("09.3 — LOCATING VALIDATION MODELS")
print("=" * 75)

MODEL_ROOT = PROJECT_ROOT / "models"

print(f"\nModel directory:")
print(MODEL_ROOT)


# ----------------------------------------------------------------------------
# 1. List all saved model files
# ----------------------------------------------------------------------------

model_files = sorted(
    MODEL_ROOT.rglob("*")
)

model_files = [
    p for p in model_files
    if p.is_file()
]

print(
    f"\nFound {len(model_files)} file(s) in models/:"
)

for path in model_files:
    print(
        " -",
        path.relative_to(PROJECT_ROOT)
    )


# ----------------------------------------------------------------------------
# 2. Identify final mapping models
# ----------------------------------------------------------------------------

rf_mapping_2003 = (
    MODEL_ROOT / "rf_final_mapping_2003.joblib"
)

rf_mapping_2025 = (
    MODEL_ROOT / "rf_final_mapping_2025.joblib"
)

print("\n" + "-" * 75)
print("FINAL MAPPING MODELS")
print("-" * 75)

for path in [
    rf_mapping_2003,
    rf_mapping_2025
]:

    print(
        f"{path.name}: "
        f"{'FOUND' if path.exists() else 'MISSING'}"
    )


# ----------------------------------------------------------------------------
# 3. Search specifically for evaluation / tuned models
# ----------------------------------------------------------------------------

evaluation_keywords = [
    "eval",
    "evaluation",
    "tuned",
    "validation",
    "cv"
]

evaluation_candidates = []

for path in model_files:

    filename = path.name.lower()

    if any(
        keyword in filename
        for keyword in evaluation_keywords
    ):
        evaluation_candidates.append(path)


print("\n" + "-" * 75)
print("POSSIBLE EVALUATION / TUNED MODELS")
print("-" * 75)

if evaluation_candidates:

    for path in evaluation_candidates:
        print(
            " -",
            path.relative_to(PROJECT_ROOT)
        )

else:

    print("No evaluation/tuned model files found.")


# ----------------------------------------------------------------------------
# 4. Check expected XGBoost evaluation model names
# ----------------------------------------------------------------------------

expected_xgb_models = {
    2003: MODEL_ROOT / "xgb_tuned_evaluation_2003.joblib",
    2025: MODEL_ROOT / "xgb_tuned_evaluation_2025.joblib"
}

print("\n" + "-" * 75)
print("EXPECTED XGBOOST EVALUATION MODELS")
print("-" * 75)

for year, path in expected_xgb_models.items():

    print(
        f"{year}: "
        f"{'FOUND' if path.exists() else 'MISSING'}"
    )


# ----------------------------------------------------------------------------
# 5. Important validation rule
# ----------------------------------------------------------------------------

print("\n" + "=" * 75)
print("VALIDATION MODEL RULE")
print("=" * 75)

print(
    "\nFinal mapping RF models will NOT be used as "
    "withheld-test validation models."
)

print(
    "They were trained on the complete labelled dataset."
)

print(
    "\nThis step only locates previously saved evaluation models."
)

print(
    "No model is loaded, retrained, or evaluated yet."
)

print("\n✓ STEP 09.3 MODEL LOCATION CHECK COMPLETE")

09.3 — LOCATING VALIDATION MODELS

Model directory:
d:\Projects\GeoAI-Flood-Susceptibility\models

Found 5 file(s) in models/:
 - models\model_metadata.json
 - models\rf_final_mapping_2003.joblib
 - models\rf_final_mapping_2025.joblib
 - models\xgb_final_mapping_2003.joblib
 - models\xgb_final_mapping_2025.joblib

---------------------------------------------------------------------------
FINAL MAPPING MODELS
---------------------------------------------------------------------------
rf_final_mapping_2003.joblib: FOUND
rf_final_mapping_2025.joblib: FOUND

---------------------------------------------------------------------------
POSSIBLE EVALUATION / TUNED MODELS
---------------------------------------------------------------------------
No evaluation/tuned model files found.

---------------------------------------------------------------------------
EXPECTED XGBOOST EVALUATION MODELS
---------------------------------------------------------------------------
2003: MISSING
2025: MISS

## 09.4 Validation Random Forest Training

The tuned Random Forest models selected in Notebook 08 were not saved as
separate evaluation-model files. They were retained in memory and the
subsequently saved Random Forest models were the final all-data mapping
models.

Therefore, the tuned Random Forest evaluation models are reproduced here
using the exact hyperparameters selected in Notebook 08.

For each year, the model is trained only on the spatial training cells,
with the NW spatial blocks completely excluded from model fitting.

The withheld NW region remains untouched until prediction and evaluation.

The reproduced models use:

### 2003
- n_estimators = 400
- max_depth = 20
- min_samples_leaf = 5
- max_features = sqrt
- class_weight = balanced
- random_state = 42

### 2025
- n_estimators = 600
- max_depth = None
- min_samples_leaf = 5
- max_features = sqrt
- class_weight = balanced
- random_state = 42

No model selection, hyperparameter tuning, resampling, or threshold
optimization is performed using the withheld NW test region.

In [9]:
# ============================================================================
# 09.4 — REPRODUCE TUNED RANDOM FOREST VALIDATION MODELS
# ============================================================================

from sklearn.ensemble import RandomForestClassifier
import joblib
import numpy as np

print("=" * 75)
print("09.4 — REPRODUCING TUNED RANDOM FOREST VALIDATION MODELS")
print("=" * 75)


# ----------------------------------------------------------------------------
# 1. Define the final predictor set explicitly
# ----------------------------------------------------------------------------
#
# These are the nine predictors used by the final models in Notebook 08.
# We define them here because Notebook 09 is a separate notebook and does
# not inherit Python variables from Notebook 08.
# ----------------------------------------------------------------------------

FINAL_PREDICTORS = [
    "Elevation",
    "Slope",
    "Flow_Accumulation",
    "River_Distance",
    "Drainage_Density",
    "Clay",
    "Sand",
    "Rainfall",
    "LULC"
]

TARGET = "Flood"

print("\nPredictors:")
for i, predictor in enumerate(FINAL_PREDICTORS, start=1):
    print(f"  {i}. {predictor}")

print(f"\nTarget: {TARGET}")


# ----------------------------------------------------------------------------
# 2. Verify that all required columns exist
# ----------------------------------------------------------------------------

for year, df in [
    (2003, modelling_2003_split),
    (2025, modelling_2025_split)
]:

    missing_predictors = [
        col
        for col in FINAL_PREDICTORS
        if col not in df.columns
    ]

    assert not missing_predictors, (
        f"{year}: missing predictor columns: "
        f"{missing_predictors}"
    )

    assert TARGET in df.columns, (
        f"{year}: target column '{TARGET}' is missing."
    )


print("\n✓ All predictors and target columns verified.")


# ----------------------------------------------------------------------------
# 3. Exact tuned RF parameters from Notebook 08
# ----------------------------------------------------------------------------

RF_VALIDATION_PARAMS = {
    2003: {
        "n_estimators": 400,
        "max_depth": 20,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs": -1
    },

    2025: {
        "n_estimators": 600,
        "max_depth": None,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs": -1
    }
}


# ----------------------------------------------------------------------------
# 4. Validation-model output directory
# ----------------------------------------------------------------------------

VALIDATION_MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "validation"
)

VALIDATION_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nValidation model directory:")
print(VALIDATION_MODEL_DIR)


# ----------------------------------------------------------------------------
# 5. Train validation RF models
# ----------------------------------------------------------------------------

rf_validation_models = {}


for year in [2003, 2025]:

    print("\n" + "=" * 75)
    print(f"YEAR {year}")
    print("=" * 75)

    # ------------------------------------------------------------------------
    # Retrieve spatially separated modelling data
    # ------------------------------------------------------------------------

    df = (
        modelling_2003_split
        if year == 2003
        else modelling_2025_split
    ).copy()

    train_df = df[
        df["dataset"] == "train"
    ].copy()

    test_df = df[
        df["dataset"] == "test"
    ].copy()


    # ------------------------------------------------------------------------
    # Prepare predictors and target
    # ------------------------------------------------------------------------

    X_train = train_df[
        FINAL_PREDICTORS
    ].copy()

    y_train = train_df[
        TARGET
    ].astype(int).copy()

    X_test = test_df[
        FINAL_PREDICTORS
    ].copy()

    y_test = test_df[
        TARGET
    ].astype(int).copy()


    # ------------------------------------------------------------------------
    # Integrity checks
    # ------------------------------------------------------------------------

    assert len(X_train) == len(y_train)
    assert len(X_test) == len(y_test)

    assert y_train.nunique() == 2
    assert y_test.nunique() == 2

    assert np.isfinite(
        X_train.to_numpy(dtype=float)
    ).all()

    assert np.isfinite(
        X_test.to_numpy(dtype=float)
    ).all()


    # ------------------------------------------------------------------------
    # Print spatial split
    # ------------------------------------------------------------------------

    print("\nSpatial data")
    print("-" * 60)

    print(f"Training cells       : {len(train_df):,}")
    print(f"Testing cells        : {len(test_df):,}")

    print(
        f"Training flood       : "
        f"{int(y_train.sum()):,}"
    )

    print(
        f"Training non-flood   : "
        f"{int((y_train == 0).sum()):,}"
    )

    print(
        f"Testing flood        : "
        f"{int(y_test.sum()):,}"
    )

    print(
        f"Testing non-flood    : "
        f"{int((y_test == 0).sum()):,}"
    )


    # ------------------------------------------------------------------------
    # Create RF
    # ------------------------------------------------------------------------

    params = RF_VALIDATION_PARAMS[year]

    print("\nRF parameters")
    print("-" * 60)

    for key, value in params.items():
        print(f"{key:18s}: {value}")


    model = RandomForestClassifier(
        **params
    )


    # ------------------------------------------------------------------------
    # Train ONLY on spatial training cells
    # ------------------------------------------------------------------------

    print("\nTraining Random Forest...")

    model.fit(
        X_train,
        y_train
    )

    rf_validation_models[year] = model

    print("✓ Training completed.")


    # ------------------------------------------------------------------------
    # Save validation model
    # ------------------------------------------------------------------------

    model_path = (
        VALIDATION_MODEL_DIR
        / f"rf_validation_{year}.joblib"
    )

    joblib.dump(
        model,
        model_path
    )

    print(
        f"✓ Saved validation model:"
    )

    print(
        f"  {model_path}"
    )


# ----------------------------------------------------------------------------
# 6. Verify saved models
# ----------------------------------------------------------------------------

print("\n" + "=" * 75)
print("VALIDATION MODEL FILE CHECK")
print("=" * 75)

for year in [2003, 2025]:

    model_path = (
        VALIDATION_MODEL_DIR
        / f"rf_validation_{year}.joblib"
    )

    print(
        f"{year}: "
        f"{'FOUND' if model_path.exists() else 'MISSING'}"
    )

    assert model_path.exists()


print("\n" + "=" * 75)
print("✓ STEP 09.4 COMPLETE")
print("=" * 75)

print(
    "\nThe tuned RF validation models were reproduced using "
    "only the spatial training cells."
)

print(
    "The NW test cells were not used during model fitting."
)

09.4 — REPRODUCING TUNED RANDOM FOREST VALIDATION MODELS

Predictors:
  1. Elevation
  2. Slope
  3. Flow_Accumulation
  4. River_Distance
  5. Drainage_Density
  6. Clay
  7. Sand
  8. Rainfall
  9. LULC

Target: Flood

✓ All predictors and target columns verified.

Validation model directory:
d:\Projects\GeoAI-Flood-Susceptibility\models\validation

YEAR 2003

Spatial data
------------------------------------------------------------
Training cells       : 32,982
Testing cells        : 14,541
Training flood       : 56
Training non-flood   : 32,926
Testing flood        : 43
Testing non-flood    : 14,498

RF parameters
------------------------------------------------------------
n_estimators      : 400
max_depth         : 20
min_samples_leaf  : 5
max_features      : sqrt
class_weight      : balanced
random_state      : 42
n_jobs            : -1

Training Random Forest...
✓ Training completed.
✓ Saved validation model:
  d:\Projects\GeoAI-Flood-Susceptibility\models\validation\rf_validat

## 09.5 RF Prediction on the Spatially Withheld Test Region

The validation Random Forest models trained in Step 09.4 are applied to the
spatially withheld NW region for 2003 and 2025.

Only the NW test cells are used for prediction. These cells were excluded
from model fitting and therefore provide the spatially withheld evaluation
dataset.

The predicted value is the Random Forest probability-like score for the
positive flood class (`class = 1`). This continuous score is treated as a
relative flood susceptibility score for ranking purposes and is not
interpreted as a calibrated probability.

No threshold-based classification or threshold optimization is performed
in this step.

The cell-level predictions are saved to `outputs/analysis_output` for
subsequent validation analysis.

In [10]:
# ============================================================================
# 09.5 — RF PREDICTION ON THE SPATIALLY WITHHELD NW TEST REGION
# ============================================================================

import pandas as pd
import numpy as np
import joblib

print("=" * 75)
print("09.5 — RF PREDICTION ON WITHHELD NW TEST REGION")
print("=" * 75)


# ----------------------------------------------------------------------------
# 1. Ensure output directory exists
# ----------------------------------------------------------------------------

ANALYSIS_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ----------------------------------------------------------------------------
# 2. Storage for predictions
# ----------------------------------------------------------------------------

rf_validation_predictions = {}


# ----------------------------------------------------------------------------
# 3. Generate predictions for each year
# ----------------------------------------------------------------------------

for year in [2003, 2025]:

    print("\n" + "=" * 75)
    print(f"YEAR {year}")
    print("=" * 75)


    # ------------------------------------------------------------------------
    # Retrieve the correct validation dataset
    # ------------------------------------------------------------------------

    df = (
        modelling_2003_split
        if year == 2003
        else modelling_2025_split
    ).copy()

    test_df = df[
        df["dataset"] == "test"
    ].copy()


    # ------------------------------------------------------------------------
    # Retrieve validation model
    # ------------------------------------------------------------------------

    model_path = (
        VALIDATION_MODEL_DIR
        / f"rf_validation_{year}.joblib"
    )

    assert model_path.exists(), (
        f"Validation model missing: {model_path}"
    )

    model = joblib.load(
        model_path
    )


    # ------------------------------------------------------------------------
    # Prepare predictors
    # ------------------------------------------------------------------------

    X_test = test_df[
        FINAL_PREDICTORS
    ].copy()

    y_test = test_df[
        TARGET
    ].astype(int).copy()


    # ------------------------------------------------------------------------
    # Predict flood-class score
    # ------------------------------------------------------------------------

    print("\nGenerating withheld-region predictions...")

    probabilities = model.predict_proba(
        X_test
    )

    # Identify the column corresponding to class 1
    class_1_index = list(
        model.classes_
    ).index(1)

    susceptibility_score = probabilities[
        :, class_1_index
    ]


    # ------------------------------------------------------------------------
    # Build prediction table
    # ------------------------------------------------------------------------

    prediction_df = test_df[
        [
            "year",
            "row",
            "col",
            "x_utm",
            "y_utm",
            "Flood"
        ]
    ].copy()

    prediction_df[
        "RF_Susceptibility"
    ] = susceptibility_score

    prediction_df[
        "Observed_Flood"
    ] = y_test.values


    # ------------------------------------------------------------------------
    # Integrity checks
    # ------------------------------------------------------------------------

    assert len(prediction_df) == len(test_df)

    assert prediction_df[
        "RF_Susceptibility"
    ].notna().all()

    assert np.isfinite(
        prediction_df[
            "RF_Susceptibility"
        ].to_numpy()
    ).all()

    assert (
        prediction_df["Observed_Flood"]
        == test_df["Flood"].values
    ).all()


    # ------------------------------------------------------------------------
    # Store in memory
    # ------------------------------------------------------------------------

    rf_validation_predictions[year] = prediction_df


    # ------------------------------------------------------------------------
    # Save CSV
    # ------------------------------------------------------------------------

    output_path = (
        ANALYSIS_OUTPUT_DIR
        / f"RF_validation_predictions_{year}.csv"
    )

    prediction_df.to_csv(
        output_path,
        index=False
    )


    # ------------------------------------------------------------------------
    # Report
    # ------------------------------------------------------------------------

    print("\nPrediction summary")
    print("-" * 60)

    print(
        f"Test cells       : "
        f"{len(prediction_df):,}"
    )

    print(
        f"Observed floods  : "
        f"{int(prediction_df['Observed_Flood'].sum()):,}"
    )

    print(
        f"Observed nonflood: "
        f"{int((prediction_df['Observed_Flood'] == 0).sum()):,}"
    )

    print(
        f"Score minimum    : "
        f"{prediction_df['RF_Susceptibility'].min():.6f}"
    )

    print(
        f"Score maximum    : "
        f"{prediction_df['RF_Susceptibility'].max():.6f}"
    )

    print(
        f"Score mean       : "
        f"{prediction_df['RF_Susceptibility'].mean():.6f}"
    )

    print(
        f"Score median     : "
        f"{prediction_df['RF_Susceptibility'].median():.6f}"
    )

    print(
        f"\n✓ Saved:"
    )

    print(
        f"  {output_path}"
    )


# ----------------------------------------------------------------------------
# 4. Final output check
# ----------------------------------------------------------------------------

print("\n" + "=" * 75)
print("VALIDATION PREDICTION FILE CHECK")
print("=" * 75)

for year in [2003, 2025]:

    output_path = (
        ANALYSIS_OUTPUT_DIR
        / f"RF_validation_predictions_{year}.csv"
    )

    print(
        f"{year}: "
        f"{'FOUND' if output_path.exists() else 'MISSING'}"
    )

    assert output_path.exists()


print("\n" + "=" * 75)
print("✓ STEP 09.5 COMPLETE")
print("=" * 75)

print(
    "\nRF susceptibility scores have been generated for "
    "the spatially withheld NW test region."
)

print(
    "No validation metrics or threshold-based classification "
    "has been performed yet."
)

09.5 — RF PREDICTION ON WITHHELD NW TEST REGION

YEAR 2003

Generating withheld-region predictions...

Prediction summary
------------------------------------------------------------
Test cells       : 14,541
Observed floods  : 43
Observed nonflood: 14,498
Score minimum    : 0.000000
Score maximum    : 0.588418
Score mean       : 0.012095
Score median     : 0.000000

✓ Saved:
  d:\Projects\GeoAI-Flood-Susceptibility\outputs\analysis_output\RF_validation_predictions_2003.csv

YEAR 2025

Generating withheld-region predictions...

Prediction summary
------------------------------------------------------------
Test cells       : 14,099
Observed floods  : 599
Observed nonflood: 13,500
Score minimum    : 0.000000
Score maximum    : 0.802524
Score mean       : 0.045923
Score median     : 0.008980

✓ Saved:
  d:\Projects\GeoAI-Flood-Susceptibility\outputs\analysis_output\RF_validation_predictions_2025.csv

VALIDATION PREDICTION FILE CHECK
2003: FOUND
2025: FOUND

✓ STEP 09.5 COMPLETE

RF susce

## 09.6 RF Withheld Spatial Validation Metrics

This step evaluates the Random Forest susceptibility scores generated for the
spatially withheld NW test region.

The evaluation is performed independently for 2003 and 2025.

Continuous ranking performance is assessed using:

- ROC-AUC
- Precision–Recall AUC (PR-AUC / Average Precision)

Because the flood class is strongly imbalanced, PR-AUC is emphasized as the
primary discrimination metric, while ROC-AUC is reported as a complementary
metric.

Threshold-based diagnostics are additionally reported at the conventional
classification threshold of 0.5:

- Precision
- Recall
- F1-score
- Accuracy
- Confusion matrix

The 0.5 threshold is not optimized using the withheld test data.

No test-region observations are used for model fitting or threshold tuning.

The resulting cell-level metrics are saved to `outputs/analysis_output`.

In [11]:
# ============================================================================
# 09.6 — RF WITHHELD SPATIAL VALIDATION METRICS
# ============================================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)

import pandas as pd
import numpy as np


print("=" * 75)
print("09.6 — RF WITHHELD SPATIAL VALIDATION METRICS")
print("=" * 75)


# ----------------------------------------------------------------------------
# 1. Storage
# ----------------------------------------------------------------------------

rf_validation_metrics = []

rf_confusion_matrices = {}


# ----------------------------------------------------------------------------
# 2. Evaluate each year
# ----------------------------------------------------------------------------

for year in [2003, 2025]:

    print("\n" + "=" * 75)
    print(f"YEAR {year}")
    print("=" * 75)


    # ------------------------------------------------------------------------
    # Load prediction table already generated in Step 09.5
    # ------------------------------------------------------------------------

    prediction_path = (
        ANALYSIS_OUTPUT_DIR
        / f"RF_validation_predictions_{year}.csv"
    )

    assert prediction_path.exists(), (
        f"Prediction file missing: {prediction_path}"
    )

    prediction_df = pd.read_csv(
        prediction_path
    )


    # ------------------------------------------------------------------------
    # Extract observed labels and continuous susceptibility scores
    # ------------------------------------------------------------------------

    y_true = (
        prediction_df["Observed_Flood"]
        .astype(int)
        .to_numpy()
    )

    y_score = (
        prediction_df["RF_Susceptibility"]
        .astype(float)
        .to_numpy()
    )


    # ------------------------------------------------------------------------
    # Integrity checks
    # ------------------------------------------------------------------------

    assert len(y_true) == len(y_score)

    assert np.isfinite(y_score).all()

    assert set(np.unique(y_true)).issubset({0, 1})

    assert np.unique(y_true).size == 2


    # ------------------------------------------------------------------------
    # Continuous ranking metrics
    # ------------------------------------------------------------------------

    roc_auc = roc_auc_score(
        y_true,
        y_score
    )

    pr_auc = average_precision_score(
        y_true,
        y_score
    )


    # ------------------------------------------------------------------------
    # Threshold-based diagnostics
    #
    # IMPORTANT:
    # 0.5 is used directly.
    # No threshold is optimized on the withheld test data.
    # ------------------------------------------------------------------------

    threshold = 0.5

    y_pred = (
        y_score >= threshold
    ).astype(int)


    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )


    # ------------------------------------------------------------------------
    # Confusion matrix
    # ------------------------------------------------------------------------

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    rf_confusion_matrices[year] = cm


    # ------------------------------------------------------------------------
    # Store metrics
    # ------------------------------------------------------------------------

    rf_validation_metrics.append({
        "year": year,
        "n_test_cells": len(y_true),
        "observed_flood_cells": int(y_true.sum()),
        "observed_nonflood_cells": int((y_true == 0).sum()),
        "ROC_AUC": roc_auc,
        "PR_AUC": pr_auc,
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "F1": f1,
        "accuracy": accuracy,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    })


    # ------------------------------------------------------------------------
    # Print results
    # ------------------------------------------------------------------------

    print("\nContinuous ranking performance")
    print("-" * 60)

    print(
        f"ROC-AUC : {roc_auc:.4f}"
    )

    print(
        f"PR-AUC  : {pr_auc:.4f}"
    )


    print("\nThreshold-based performance")
    print("-" * 60)

    print(
        f"Threshold : {threshold:.2f}"
    )

    print(
        f"Precision : {precision:.4f}"
    )

    print(
        f"Recall    : {recall:.4f}"
    )

    print(
        f"F1-score  : {f1:.4f}"
    )

    print(
        f"Accuracy  : {accuracy:.4f}"
    )


    print("\nConfusion matrix")
    print("-" * 60)

    print(
        "              Predicted"
    )

    print(
        "              0       1"
    )

    print(
        f"Observed 0   {tn:5d}   {fp:5d}"
    )

    print(
        f"Observed 1   {fn:5d}   {tp:5d}"
    )


# ----------------------------------------------------------------------------
# 3. Create metrics DataFrame
# ----------------------------------------------------------------------------

rf_validation_metrics_df = pd.DataFrame(
    rf_validation_metrics
)


# ----------------------------------------------------------------------------
# 4. Save validation metrics
# ----------------------------------------------------------------------------

metrics_output_path = (
    ANALYSIS_OUTPUT_DIR
    / "RF_withheld_spatial_validation_metrics.csv"
)

rf_validation_metrics_df.to_csv(
    metrics_output_path,
    index=False
)


# ----------------------------------------------------------------------------
# 5. Display summary table
# ----------------------------------------------------------------------------

print("\n" + "=" * 75)
print("RF WITHHELD SPATIAL VALIDATION SUMMARY")
print("=" * 75)

print(
    rf_validation_metrics_df[
        [
            "year",
            "n_test_cells",
            "observed_flood_cells",
            "ROC_AUC",
            "PR_AUC",
            "precision",
            "recall",
            "F1",
            "accuracy"
        ]
    ].to_string(index=False)
)


# ----------------------------------------------------------------------------
# 6. Verify output
# ----------------------------------------------------------------------------

assert metrics_output_path.exists()

assert len(
    rf_validation_metrics_df
) == 2


print("\n" + "=" * 75)
print("✓ STEP 09.6 COMPLETE")
print("=" * 75)

print(
    f"\nSaved metrics to:"
)

print(
    metrics_output_path
)

09.6 — RF WITHHELD SPATIAL VALIDATION METRICS

YEAR 2003

Continuous ranking performance
------------------------------------------------------------
ROC-AUC : 0.6466
PR-AUC  : 0.0131

Threshold-based performance
------------------------------------------------------------
Threshold : 0.50
Precision : 0.0000
Recall    : 0.0000
F1-score  : 0.0000
Accuracy  : 0.9961

Confusion matrix
------------------------------------------------------------
              Predicted
              0       1
Observed 0   14485      13
Observed 1      43       0

YEAR 2025

Continuous ranking performance
------------------------------------------------------------
ROC-AUC : 0.7968
PR-AUC  : 0.1429

Threshold-based performance
------------------------------------------------------------
Threshold : 0.50
Precision : 0.0956
Recall    : 0.0217
F1-score  : 0.0354
Accuracy  : 0.9497

Confusion matrix
------------------------------------------------------------
              Predicted
              0       1
Obse